In [53]:
import os
import h5py
import numpy as np
import rasterio
from rasterio.transform import from_bounds, from_origin
from rasterio.warp import reproject, Resampling
import matplotlib.pyplot as plt

os.makedirs("capas", exist_ok=True)

# 1. COORDENADAS BAHIA CONCEPCION
OESTE, SUR, ESTE, NORTE = -73.60, -37.70, -72.70, -36.50
ANCHO, ALTO = 1125, 1500
DST_TRANSFORM = from_bounds(OESTE, SUR, ESTE, NORTE, ANCHO, ALTO)

#CALCULADORAS

def a_decibeles(matriz):
    #Convierte potencia lineal SAR a decibeles (dB) reales
    matriz = np.where(matriz <= 0, np.nan, matriz)
    return 10.0 * np.log10(matriz)

def estirar_contraste(db, p_min=2, p_max=98):
    #Normaliza dB entre 0 y 1 para visualización PNG
    vmin, vmax = np.nanpercentile(db, (p_min, p_max))
    norm = np.clip((db - vmin) / (vmax - vmin), 0, 1)
    return np.nan_to_num(norm, nan=0.0)


# 2. REPROYECTAR Y RECORTAR SENTINEL-1 (.tiff)

def procesar_sentinel1_biobio(ruta_vv, ruta_vh, nombre_png):
    bandas = []
    for ruta in (ruta_vv, ruta_vh):
        with rasterio.open(ruta) as src:
            gcps, gcp_crs = src.gcps
            destino = np.zeros((ALTO, ANCHO), dtype=np.float32)
            reproject(
                source=rasterio.band(src, 1),
                destination=destino,
                gcps=gcps,
                src_crs=gcp_crs,
                src_nodata=0,
                dst_transform=DST_TRANSFORM,
                dst_crs="EPSG:4326",
                dst_nodata=0,
                resampling=Resampling.bilinear,
            )
            bandas.append(destino)

    vv, vh = bandas
    valido = (vv > 0) & (vh > 0)

    db_vv = a_decibeles(vv)
    db_vh = a_decibeles(vh)

    r = estirar_contraste(db_vv)
    g = estirar_contraste(db_vh)
    b = np.clip(r / (g + 0.01), 0, 1)
    alfa = np.where(valido, 1.0, 0.0)

    ruta_salida = os.path.join("capas", nombre_png)
    plt.imsave(ruta_salida, np.dstack((r, g, b, alfa)))
    print(f"OK Sentinel-1 recortado -> {ruta_salida}")
    return db_vv, db_vh, valido

# 3. REPROYECTAR Y RECORTAR NISAR L2 GCOV (.h5)

def procesar_nisar_biobio(ruta_h5, nombre_png):
    ruta_grid = "/science/LSAR/GCOV/grids/frequencyA"
    with h5py.File(ruta_h5, "r") as f:
        grid = f[ruta_grid]
        x = grid["xCoordinates"][()]
        y = grid["yCoordinates"][()]
        epsg = int(grid["projection"][()])

        # Submuestreo 1:2 para velocidad manteniendo alta nitidez
        paso = 2
        hh_raw = grid["HHHH"][::paso, ::paso].astype(np.float32)
        hv_raw = grid["HVHV"][::paso, ::paso].astype(np.float32)

        dx = abs(x[1] - x[0]) * paso
        dy = abs(y[1] - y[0]) * paso
        src_transform = from_origin(x[0] - dx / 2, y[0] + dy / 2, dx, dy)
        src_crs = f"EPSG:{epsg}"

    bandas_dest = []
    for mat in (hh_raw, hv_raw):
        destino = np.zeros((ALTO, ANCHO), dtype=np.float32)
        reproject(
            source=mat,
            destination=destino,
            src_transform=src_transform,
            src_crs=src_crs,
            src_nodata=0,
            dst_transform=DST_TRANSFORM,
            dst_crs="EPSG:4326",
            dst_nodata=0,
            resampling=Resampling.bilinear,
        )
        bandas_dest.append(destino)

    hh, hv = bandas_dest
    valido = (hh > 0) & (hv > 0)

    db_hh = a_decibeles(hh)
    db_hv = a_decibeles(hv)

    r = estirar_contraste(db_hh)
    g = estirar_contraste(db_hv)
    b = r.copy()
    alfa = np.where(valido, 1.0, 0.0)

    ruta_salida = os.path.join("capas", nombre_png)
    plt.imsave(ruta_salida, np.dstack((r, g, b, alfa)))
    print(f"OK NISAR recortado -> {ruta_salida}")
    return db_hh, db_hv, valido

In [54]:
# SENTINEL-1 (Junio 2024) 
vv_jun01, vh_jun01, val_jun01 = procesar_sentinel1_biobio(
    r"SAR-images/S1A_IW_GRDH_1SDV_20240601T234345_20240601T234409_054136_069554_E06C/s1a-iw-grd-vh-20240601t234345-20240601t234409-054136-069554-002.tiff",
    r"SAR-images/S1A_IW_GRDH_1SDV_20240601T234345_20240601T234409_054136_069554_E06C/s1a-iw-grd-vv-20240601t234345-20240601t234409-054136-069554-001.tiff",
    "s1_antes.png"
)

vv_jun13, vh_jun13, val_jun13 = procesar_sentinel1_biobio(
    r"SAR-images/S1A_IW_GRDH_1SDV_20240613T234344_20240613T234409_054311_069B60_4D1C/s1a-iw-grd-vh-20240613t234344-20240613t234409-054311-069b60-002.tiff",
    r"SAR-images/S1A_IW_GRDH_1SDV_20240613T234344_20240613T234409_054311_069B60_4D1C/s1a-iw-grd-vv-20240613t234344-20240613t234409-054311-069b60-001.tiff",
    "s1_desborde.png"
)


# NISAR (Septiembre 2026) 
hh_sep18, hv_sep18, val_sep18 = procesar_nisar_biobio(
    r"NISAR-images/NISAR_L2_PR_GCOV_031_003_A_159_2005_DHDH_A_20260918T105143_20260918T105220_P05023_F_F_J_001.h5",
    "nisar_antes.png"
)

hh_sep30, hv_sep30, val_sep30 = procesar_nisar_biobio(
    r"NISAR-images/NISAR_L2_PR_GCOV_032_003_A_159_2005_DHDH_A_20260930T105144_20260930T105220_P05023_N_F_J_001.h5",
    "nisar_desborde.png"
)


OK Sentinel-1 recortado -> capas\s1_antes.png
OK Sentinel-1 recortado -> capas\s1_desborde.png
OK NISAR recortado -> capas\nisar_antes.png
OK NISAR recortado -> capas\nisar_desborde.png


In [55]:
def generar_mapa_cambio_rgb(db_antes, val_antes, db_desborde, val_desborde, nombre_salida):
    """
    Genera un compuesto multitemporal RGB (estándar ESA/NASA):
    - Canal Rojo = Antes del temporal
    - Canales Verde y Azul = Durante/Post desborde
    """
    zona_comun = val_antes & val_desborde

    # Aplicar exactamente la misma escala de brillo a ambas fechas para que la comparación sea fiel
    datos_combinados = np.concatenate([db_antes[zona_comun], db_desborde[zona_comun]])
    vmin, vmax = np.nanpercentile(datos_combinados, (2, 98))

    norm_antes = np.nan_to_num(np.clip((db_antes - vmin) / (vmax - vmin), 0, 1), nan=0.0)
    norm_desborde = np.nan_to_num(np.clip((db_desborde - vmin) / (vmax - vmin), 0, 1), nan=0.0)

    r = norm_antes
    g = norm_desborde
    b = norm_desborde
    alfa = np.where(zona_comun, 1.0, 0.0)

    ruta_salida = os.path.join("capas", nombre_salida)
    # Reemplaza r, g, b en la llamada o invierte las variables al vuelo:
    plt.imsave(ruta_salida, np.dstack((r, g, b, alfa)))    
    print(f"¡Listo! Mapa de cambio guardado en -> {ruta_salida}")

# Ejecutar para Sentinel-1 (Junio 2024) y NISAR (Septiembre 2026)
generar_mapa_cambio_rgb(vv_jun01, val_jun01, vv_jun13, val_jun13, "s1_cambio_rgb.png")

def generar_mapa_cambio_nisar_deltadb(db_seco, val_seco, db_inundado, val_inundado, nombre_salida):
    zona_comun = val_seco & val_inundado
    diferencia = db_inundado - db_seco

    # El mar permanente es el 36% más oscuro en la fecha seca; el agua inundada llega hasta el 45%
    limite_mar = np.nanpercentile(db_seco[zona_comun], 36)
    limite_agua = np.nanpercentile(db_inundado[zona_comun], 41)

    # 1. Rojo: era tierra seca el 30 Sep (> limite_mar) y pasó a agua oscura el 18 Sep (< limite_agua)
    agua_abierta = (diferencia < -1.5) & (db_seco > limite_mar) & (db_inundado < limite_agua)

    # 2. Cian: subida brusca de señal en vegetación (doble rebote Banda L)
    agua_vegetacion = (diferencia > 3.5) & (db_inundado >= limite_agua)

    base_gris = np.nan_to_num(np.clip(
        (db_seco - np.nanpercentile(db_seco[zona_comun], 2)) /
        (np.nanpercentile(db_seco[zona_comun], 98) - np.nanpercentile(db_seco[zona_comun], 2)),
        0.2, 0.8), nan=0.5)

    r = np.where(agua_abierta, 1.0, np.where(agua_vegetacion, 0.10, base_gris))
    g = np.where(agua_abierta, 0.15, np.where(agua_vegetacion, 0.90, base_gris))
    b = np.where(agua_abierta, 0.15, np.where(agua_vegetacion, 0.95, base_gris))
    alfa = np.where(zona_comun, 1.0, 0.0)

    ruta_salida = os.path.join("capas", nombre_salida)
    plt.imsave(ruta_salida, np.dstack((r, g, b, alfa)))
    print(f"¡NISAR final listo! -> {ruta_salida}")

generar_mapa_cambio_nisar_deltadb(hv_sep18, val_sep18, hv_sep30, val_sep30, "nisar_cambio_rgb.png")

¡Listo! Mapa de cambio guardado en -> capas\s1_cambio_rgb.png
¡NISAR final listo! -> capas\nisar_cambio_rgb.png
